# Data Preparation and Exploratory Data Analysis
## RecoMart Recommendation Pipeline

This notebook demonstrates:
- **Data Cleaning**: Handling missing values and duplicates
- **Preprocessing**: Encoding categorical variables and normalizing numerical features
- **Exploratory Analysis**: Understanding interaction patterns, item popularity, and sparsity

**Author**: Data Preparation Team  
**Date**: 2026-07-18

## Section 1: Import Required Libraries

Import necessary libraries for data manipulation, preprocessing, and visualization.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
import warnings

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preparation.data_cleaner import DataCleaner
from src.preparation.data_preprocessor import DataPreprocessor
from src.preparation.exploratory_analysis import ExploratoryAnalyzer
from src.utils.logger import logger

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All libraries imported successfully!")
print(f"✓ Project root: {project_root}")

## Section 2: Load and Inspect Dataset

Load the datasets and examine their basic properties including shape, data types, and sample data.

In [ ]:
# Define dataset paths
dataset_dir = project_root / "dataset"

# Load all datasets
df_movies = pd.read_csv(dataset_dir / "movies.csv")
df_ratings = pd.read_csv(dataset_dir / "ratings.csv")
df_tags = pd.read_csv(dataset_dir / "tags.csv")
df_links = pd.read_csv(dataset_dir / "links.csv")

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

# Movies dataset
print("\n📽️ MOVIES Dataset")
print(f"Shape: {df_movies.shape}")
print(f"Data Types:\n{df_movies.dtypes}")
print(f"\nFirst 5 rows:")
print(df_movies.head())
print(f"\nMissing values:\n{df_movies.isna().sum()}")

# Ratings dataset
print("\n" + "=" * 80)
print("\n⭐ RATINGS Dataset")
print(f"Shape: {df_ratings.shape}")
print(f"Data Types:\n{df_ratings.dtypes}")
print(f"\nFirst 5 rows:")
print(df_ratings.head())
print(f"\nMissing values:\n{df_ratings.isna().sum()}")
print(f"Rating statistics:\n{df_ratings['rating'].describe()}")

# Tags dataset
print("\n" + "=" * 80)
print("\n🏷️ TAGS Dataset")
print(f"Shape: {df_tags.shape}")
print(f"Data Types:\n{df_tags.dtypes}")
print(f"\nFirst 5 rows:")
print(df_tags.head())
print(f"\nMissing values:\n{df_tags.isna().sum()}")

# Links dataset
print("\n" + "=" * 80)
print("\n🔗 LINKS Dataset")
print(f"Shape: {df_links.shape}")
print(f"Data Types:\n{df_links.dtypes}")
print(f"\nFirst 5 rows:")
print(df_links.head())
print(f"\nMissing values:\n{df_links.isna().sum()}")

print("\n" + "=" * 80)

## Section 3: Handle Missing User-Item Interactions

Identify and handle missing values in user-item interactions using appropriate strategies.

In [ ]:
# Initialize data cleaner
cleaner = DataCleaner()

# Define missing value strategies
missing_strategies = {
    "ratings": {
        "userId": "drop",
        "movieId": "drop",
        "rating": "drop",
        "timestamp": "drop",
    },
    "tags": {
        "userId": "drop",
        "movieId": "drop",
        "tag": "drop",
        "timestamp": "drop",
    },
    "links": {
        "movieId": "drop",
        "imdbId": "drop",
        "tmdbId": "drop",
    },
    "movies": {
        "movieId": "drop",
        "title": "drop",
        "genres": "drop",
    },
}

# Clean datasets
print("CLEANING DATASETS")
print("=" * 80)

df_ratings_clean, ratings_clean_report = cleaner.clean_csv(
    dataset_dir / "ratings.csv", 
    "ratings",
    missing_strategies["ratings"]
)

df_tags_clean, tags_clean_report = cleaner.clean_csv(
    dataset_dir / "tags.csv",
    "tags",
    missing_strategies["tags"]
)

df_links_clean, links_clean_report = cleaner.clean_csv(
    dataset_dir / "links.csv",
    "links",
    missing_strategies["links"]
)

df_movies_clean, movies_clean_report = cleaner.clean_csv(
    dataset_dir / "movies.csv",
    "movies",
    missing_strategies["movies"]
)

# Print cleaning reports
print("\n📊 RATINGS Cleaning Report")
print(f"  Original rows: {ratings_clean_report['original_rows']}")
print(f"  Final rows: {ratings_clean_report['final_rows']}")
print(f"  Rows dropped: {ratings_clean_report['rows_dropped']}")

print("\n📊 TAGS Cleaning Report")
print(f"  Original rows: {tags_clean_report['original_rows']}")
print(f"  Final rows: {tags_clean_report['final_rows']}")
print(f"  Rows dropped: {tags_clean_report['rows_dropped']}")

print("\n📊 LINKS Cleaning Report")
print(f"  Original rows: {links_clean_report['original_rows']}")
print(f"  Final rows: {links_clean_report['final_rows']}")
print(f"  Rows dropped: {links_clean_report['rows_dropped']}")
print(f"  Operations: {[op['operation'] for op in links_clean_report['operations']]}")

print("\n" + "=" * 80)
print("✓ Data cleaning completed!")

## Section 4: Encode Categorical Attributes

Encode categorical variables like genres and tags using label encoding.

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

print("ENCODING CATEGORICAL VARIABLES")
print("=" * 80)

# Preprocess movies with genre encoding
print("\n🎬 Encoding MOVIES data...")
df_movies_processed, movies_encoding_report = preprocessor.preprocess_dataset(
    df_movies_clean,
    "movies",
    categorical_columns=["genres"],
    numerical_columns=["movieId"]
)

print(f"  Original shape: {df_movies_clean.shape}")
print(f"  Processed shape: {df_movies_processed.shape}")
print(f"  Encoding operations:")
for col, report in movies_encoding_report["encoding_operations"].items():
    print(f"    - {col}: {report['method']} ({report['unique_values']} unique)")

# Preprocess tags with tag encoding
print("\n🏷️ Encoding TAGS data...")
df_tags_processed, tags_encoding_report = preprocessor.preprocess_dataset(
    df_tags_clean,
    "tags",
    categorical_columns=["tag"],
    numerical_columns=["userId", "movieId"]
)

print(f"  Original shape: {df_tags_clean.shape}")
print(f"  Processed shape: {df_tags_processed.shape}")
print(f"  Encoding operations:")
for col, report in tags_encoding_report["encoding_operations"].items():
    print(f"    - {col}: {report['method']} ({report['unique_values']} unique)")

# Display encoded samples
print("\n" + "=" * 80)
print("Sample of encoded MOVIES data:")
print(df_movies_processed.head())

print("\nSample of encoded TAGS data:")
print(df_tags_processed.head())

print("\n✓ Categorical encoding completed!")

## Section 5: Normalize Numerical Variables

Normalize numerical features like ratings and timestamps using MinMax scaling.

In [ ]:
print("NORMALIZING NUMERICAL VARIABLES")
print("=" * 80)

# Normalize ratings data
print("\n⭐ Normalizing RATINGS data...")
df_ratings_processed, ratings_norm_report = preprocessor.preprocess_dataset(
    df_ratings_clean,
    "ratings",
    categorical_columns=[],
    numerical_columns=["userId", "movieId", "rating", "timestamp"],
    normalize_method="minmax"
)

print(f"  Original shape: {df_ratings_clean.shape}")
print(f"  Processed shape: {df_ratings_processed.shape}")
print(f"  Normalization operations:")
for col, report in ratings_norm_report["normalization_operations"].items():
    print(f"    - {col}: {report['method']}")
    print(f"      Original range: [{report['original_min']:.2f}, {report['original_max']:.2f}]")
    print(f"      Normalized range: [{report['normalized_min']:.2f}, {report['normalized_max']:.2f}]")

# Normalize links data
print("\n🔗 Normalizing LINKS data...")
df_links_processed, links_norm_report = preprocessor.preprocess_dataset(
    df_links_clean,
    "links",
    categorical_columns=[],
    numerical_columns=["movieId", "imdbId", "tmdbId"],
    normalize_method="minmax"
)

print(f"  Original shape: {df_links_clean.shape}")
print(f"  Processed shape: {df_links_processed.shape}")

# Display normalized samples
print("\n" + "=" * 80)
print("Sample of normalized RATINGS data:")
print(df_ratings_processed.head())
print("\nRATINGS statistics after normalization:")
print(df_ratings_processed.describe())

print("\n✓ Numerical normalization completed!")

## Section 6: Exploratory Data Analysis - Interaction Distributions

Analyze the distribution of user-item interactions to understand interaction patterns.

In [ ]:
# Initialize analyzer
analyzer = ExploratoryAnalyzer()

print("EXPLORATORY DATA ANALYSIS - INTERACTION DISTRIBUTIONS")
print("=" * 80)

# Analyze ratings distribution
ratings_analysis = analyzer.analyze_dataset(df_ratings_clean, "ratings")

print("\n📊 RATINGS Distribution Analysis")
print(f"  Mean rating: {df_ratings_clean['rating'].mean():.3f}")
print(f"  Median rating: {df_ratings_clean['rating'].median():.3f}")
print(f"  Std Dev: {df_ratings_clean['rating'].std():.3f}")
print(f"  Skewness: {df_ratings_clean['rating'].skew():.3f}")
print(f"  Kurtosis: {df_ratings_clean['rating'].kurtosis():.3f}")

print("\n  Rating value counts:")
print(df_ratings_clean['rating'].value_counts().sort_index())

# Plot rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_ratings_clean['rating'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_title('Rating Distribution - Histogram', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df_ratings_clean['rating'], vert=True)
axes[1].set_title('Rating Distribution - Box Plot', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Rating Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Interaction distribution analysis completed!")

## Section 7: Exploratory Data Analysis - Item Popularity

Examine item popularity by analyzing interaction counts and identifying top-rated items.

In [ ]:
print("ITEM POPULARITY ANALYSIS")
print("=" * 80)

# Calculate item popularity
item_popularity = df_ratings_clean['movieId'].value_counts()
item_ratings = df_ratings_clean.groupby('movieId')['rating'].agg(['mean', 'count', 'std'])
item_ratings.columns = ['avg_rating', 'num_ratings', 'rating_std']
item_ratings = item_ratings.sort_values('num_ratings', ascending=False)

print("\n📊 Top 10 Most Popular Items (by number of ratings)")
print(item_ratings.head(10))

print("\n📊 Top 10 Highest Rated Items (with at least 20 ratings)")
top_rated = item_ratings[item_ratings['num_ratings'] >= 20].sort_values('avg_rating', ascending=False)
print(top_rated.head(10))

# Merge with movie titles
top_popular_ids = item_ratings.head(15).index
top_popular_with_titles = df_movies_clean[df_movies_clean['movieId'].isin(top_popular_ids)].copy()
top_popular_with_titles = top_popular_with_titles.merge(
    item_ratings.reset_index().rename(columns={'movieId': 'movieId'}),
    on='movieId'
).sort_values('num_ratings', ascending=False)

print("\n🎬 Top 15 Popular Movies with Titles")
print(top_popular_with_titles[['movieId', 'title', 'avg_rating', 'num_ratings']].to_string())

# Plot item popularity
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Distribution of item popularity
axes[0, 0].hist(item_popularity.values, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Item Popularity', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Ratings per Item')
axes[0, 0].set_ylabel('Number of Items')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Top 20 items
top_20 = item_ratings.head(20).sort_values('num_ratings')
axes[0, 1].barh(range(len(top_20)), top_20['num_ratings'].values, color='coral')
axes[0, 1].set_yticks(range(len(top_20)))
axes[0, 1].set_yticklabels([f"Item {mid}" for mid in top_20.index], fontsize=8)
axes[0, 1].set_title('Top 20 Most Popular Items', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Ratings')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Plot 3: Average rating vs popularity
axes[1, 0].scatter(item_ratings['num_ratings'], item_ratings['avg_rating'], alpha=0.5, s=30)
axes[1, 0].set_title('Item Popularity vs Average Rating', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Ratings (log scale)')
axes[1, 0].set_ylabel('Average Rating')
axes[1, 0].set_xscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Rating distribution statistics
rating_stats = item_ratings[['avg_rating', 'rating_std']].describe()
axes[1, 1].axis('off')
stats_text = f"""
Item Popularity Statistics:

Total Unique Items: {len(item_ratings)}
Total Ratings: {item_ratings['num_ratings'].sum():,}

Ratings per Item:
  Min: {item_ratings['num_ratings'].min()}
  Max: {item_ratings['num_ratings'].max()}
  Mean: {item_ratings['num_ratings'].mean():.2f}
  Median: {item_ratings['num_ratings'].median():.2f}
  
Average Rating per Item:
  Min: {item_ratings['avg_rating'].min():.2f}
  Max: {item_ratings['avg_rating'].max():.2f}
  Mean: {item_ratings['avg_rating'].mean():.2f}
  
Rating Std Dev:
  Mean: {item_ratings['rating_std'].mean():.2f}
  Max: {item_ratings['rating_std'].max():.2f}
"""
axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.show()

print("\n✓ Item popularity analysis completed!")

## Section 8: Exploratory Data Analysis - Sparsity Patterns

Calculate and visualize matrix sparsity, interaction density, and data sparsity patterns.

In [ ]:
print("SPARSITY ANALYSIS")
print("=" * 80)

# Create user-item interaction matrix
interaction_matrix = preprocessor.create_interaction_matrix(
    df_ratings_clean,
    user_col='userId',
    item_col='movieId',
    rating_col='rating'
)

# Calculate sparsity metrics
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]
filled_cells = (interaction_matrix != 0).sum().sum()
empty_cells = total_cells - filled_cells
sparsity_pct = (empty_cells / total_cells) * 100
density_pct = (filled_cells / total_cells) * 100

print(f"\n📊 Interaction Matrix Sparsity Analysis")
print(f"  Matrix shape: {interaction_matrix.shape[0]} users × {interaction_matrix.shape[1]} items")
print(f"  Total cells: {total_cells:,}")
print(f"  Filled cells: {filled_cells:,}")
print(f"  Empty cells: {empty_cells:,}")
print(f"  Sparsity: {sparsity_pct:.2f}%")
print(f"  Density: {density_pct:.2f}%")

# Calculate per-user and per-item statistics
user_interactions = (interaction_matrix != 0).sum(axis=1)
item_interactions = (interaction_matrix != 0).sum(axis=0)

print(f"\n👥 User Interaction Statistics")
print(f"  Mean interactions per user: {user_interactions.mean():.2f}")
print(f"  Median interactions per user: {user_interactions.median():.2f}")
print(f"  Min interactions: {user_interactions.min()}")
print(f"  Max interactions: {user_interactions.max()}")
print(f"  Std dev: {user_interactions.std():.2f}")

print(f"\n🎬 Item Popularity Statistics")
print(f"  Mean ratings per item: {item_interactions.mean():.2f}")
print(f"  Median ratings per item: {item_interactions.median():.2f}")
print(f"  Min ratings: {item_interactions.min()}")
print(f"  Max ratings: {item_interactions.max()}")
print(f"  Std dev: {item_interactions.std():.2f}")

# Plot sparsity
plot_path = analyzer.plot_sparsity(interaction_matrix, "user_item_interactions")
print(f"\n✓ Sparsity plot saved to: {plot_path}")

plt.show()

print("\n✓ Sparsity analysis completed!")

## Section 9: Generate Summary Visualizations

Create comprehensive summary plots including histograms, heatmaps, and distribution plots.

In [ ]:
print("GENERATING SUMMARY VISUALIZATIONS")
print("=" * 80)

# Create output directory
plots_dir = project_root / "data" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

# Plot 1: Rating distributions
print("\n📊 Generating rating distribution plots...")
analyzer.plot_distributions(df_ratings_clean, "ratings_clean", save=True)

# Plot 2: Categorical distributions (genres, tags)
print("📊 Generating categorical distribution plots...")
analyzer.plot_categorical_distributions(df_movies_clean, "movies", save=True)

# Plot 3: Correlation heatmap
print("📊 Generating correlation heatmap...")
analyzer.plot_heatmap(df_ratings_clean, "ratings", save=True)

# Plot 4: Comprehensive summary
print("📊 Creating comprehensive summary visualization...")
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Plot 4.1: Rating distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_ratings_clean['rating'], bins=25, alpha=0.7, color='steelblue', edgecolor='black')
ax1.set_title('Rating Distribution', fontweight='bold')
ax1.set_xlabel('Rating')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# Plot 4.2: User interactions per user
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(user_interactions, bins=50, alpha=0.7, color='forestgreen', edgecolor='black')
ax2.set_title('Ratings per User Distribution', fontweight='bold')
ax2.set_xlabel('Number of Ratings')
ax2.set_ylabel('Number of Users')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

# Plot 4.3: Item popularity
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(item_interactions, bins=50, alpha=0.7, color='coral', edgecolor='black')
ax3.set_title('Ratings per Item Distribution', fontweight='bold')
ax3.set_xlabel('Number of Ratings')
ax3.set_ylabel('Number of Items')
ax3.set_yscale('log')
ax3.grid(True, alpha=0.3)

# Plot 4.4: Top 15 users
ax4 = fig.add_subplot(gs[1, 0])
top_users = user_interactions.nlargest(15)
ax4.barh(range(len(top_users)), top_users.values, color='steelblue')
ax4.set_yticks(range(len(top_users)))
ax4.set_yticklabels([f"User {uid}" for uid in top_users.index], fontsize=8)
ax4.set_title('Top 15 Most Active Users', fontweight='bold')
ax4.set_xlabel('Number of Ratings')
ax4.grid(True, alpha=0.3, axis='x')

# Plot 4.5: Top 15 items
ax5 = fig.add_subplot(gs[1, 1])
top_items = item_interactions.nlargest(15)
ax5.barh(range(len(top_items)), top_items.values, color='coral')
ax5.set_yticks(range(len(top_items)))
ax5.set_yticklabels([f"Item {iid}" for iid in top_items.index], fontsize=8)
ax5.set_title('Top 15 Most Popular Items', fontweight='bold')
ax5.set_xlabel('Number of Ratings')
ax5.grid(True, alpha=0.3, axis='x')

# Plot 4.6: Sparsity pie chart
ax6 = fig.add_subplot(gs[1, 2])
sparsity_data = [filled_cells, empty_cells]
colors = ['#66c2a5', '#fc8d62']
ax6.pie(sparsity_data, labels=['Filled', 'Empty'], autopct='%1.1f%%', colors=colors)
ax6.set_title(f'Matrix Sparsity: {sparsity_pct:.2f}%', fontweight='bold')

# Plot 4.7: User-Item interactions scatter
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(user_interactions, item_interactions, alpha=0.5, s=20)
ax7.set_title('User Interactions vs Item Popularity', fontweight='bold')
ax7.set_xlabel('User Interactions (log)')
ax7.set_ylabel('Item Popularity (log)')
ax7.set_xscale('log')
ax7.set_yscale('log')
ax7.grid(True, alpha=0.3)

# Plot 4.8: Data quality summary
ax8 = fig.add_subplot(gs[2, 1:])
ax8.axis('off')
summary_text = f"""
DATA PREPARATION SUMMARY

Dataset Sizes:
  Users: {interaction_matrix.shape[0]:,}
  Items: {interaction_matrix.shape[1]:,}
  Total Ratings: {filled_cells:,}

Matrix Properties:
  Sparsity: {sparsity_pct:.2f}%
  Density: {density_pct:.2f}%
  
Rating Statistics:
  Mean: {df_ratings_clean['rating'].mean():.2f}
  Median: {df_ratings_clean['rating'].median():.2f}
  Std Dev: {df_ratings_clean['rating'].std():.2f}
  Min: {df_ratings_clean['rating'].min():.1f}
  Max: {df_ratings_clean['rating'].max():.1f}

User Behavior:
  Avg ratings/user: {user_interactions.mean():.2f}
  Max ratings/user: {user_interactions.max()}
  Min ratings/user: {user_interactions.min()}

Item Popularity:
  Avg ratings/item: {item_interactions.mean():.2f}
  Max ratings/item: {item_interactions.max()}
  Min ratings/item: {item_interactions.min()}
"""
ax8.text(0.05, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center')

plt.suptitle('Data Preparation Summary Visualization', fontsize=16, fontweight='bold', y=0.995)

# Save comprehensive summary plot
summary_plot_path = plots_dir / "comprehensive_summary.png"
plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
logger.info(f"Saved comprehensive summary: {summary_plot_path}")
plt.show()

print("\n✓ All summary visualizations generated!")

## Section 10: Export Prepared Dataset

Save the cleaned and preprocessed datasets for use in subsequent transformation and modeling.

In [ ]:
print("EXPORTING PREPARED DATASETS")
print("=" * 80)

# Create processed data directory
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned datasets
print("\n💾 Saving cleaned datasets...")
cleaned_files = cleaner.save_cleaned_data(processed_dir)
for dataset_name, file_path in cleaned_files.items():
    file_size = file_path.stat().st_size / 1024
    print(f"  ✓ {dataset_name}: {file_path} ({file_size:.2f} KB)")

# Save processed datasets
print("\n💾 Saving preprocessed datasets...")
processed_files = preprocessor.save_processed_data(processed_dir)
for dataset_name, file_path in processed_files.items():
    file_size = file_path.stat().st_size / 1024
    print(f"  ✓ {dataset_name}: {file_path} ({file_size:.2f} KB)")

# Save interaction matrix
print("\n💾 Saving user-item interaction matrix...")
matrix_file = processed_dir / "user_item_interaction_matrix.csv"
interaction_matrix.to_csv(matrix_file)
matrix_size = matrix_file.stat().st_size / (1024 * 1024)
print(f"  ✓ Interaction matrix: {matrix_file} ({matrix_size:.2f} MB)")
print(f"    Shape: {interaction_matrix.shape[0]} users × {interaction_matrix.shape[1]} items")

# Save sparsity report
print("\n💾 Saving sparsity report...")
sparsity_report = {
    "matrix_shape": interaction_matrix.shape,
    "total_cells": int(total_cells),
    "filled_cells": int(filled_cells),
    "empty_cells": int(empty_cells),
    "sparsity_percentage": float(sparsity_pct),
    "density_percentage": float(density_pct),
    "user_stats": {
        "mean_interactions": float(user_interactions.mean()),
        "median_interactions": float(user_interactions.median()),
        "min_interactions": int(user_interactions.min()),
        "max_interactions": int(user_interactions.max()),
        "std_dev": float(user_interactions.std()),
    },
    "item_stats": {
        "mean_interactions": float(item_interactions.mean()),
        "median_interactions": float(item_interactions.median()),
        "min_interactions": int(item_interactions.min()),
        "max_interactions": int(item_interactions.max()),
        "std_dev": float(item_interactions.std()),
    }
}

import json
sparsity_file = processed_dir / "sparsity_report.json"
with open(sparsity_file, 'w') as f:
    json.dump(sparsity_report, f, indent=2)
print(f"  ✓ Sparsity report: {sparsity_file}")

# Create data preparation summary
print("\n📊 Data Preparation Summary")
summary_stats = {
    "cleaned_datasets": list(cleaned_files.keys()),
    "processed_datasets": list(processed_files.keys()),
    "total_files_generated": len(cleaned_files) + len(processed_files) + 2,
    "plots_generated": 4,  # distributions, heatmap, sparsity, summary
    "statistics": {
        "total_users": int(interaction_matrix.shape[0]),
        "total_items": int(interaction_matrix.shape[1]),
        "total_interactions": int(filled_cells),
        "matrix_sparsity_pct": float(sparsity_pct),
        "avg_interactions_per_user": float(user_interactions.mean()),
        "avg_popularity_per_item": float(item_interactions.mean()),
    }
}

summary_file = processed_dir / "data_preparation_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"  ✓ Preparation summary: {summary_file}")

print("\n" + "=" * 80)
print("✅ DATA PREPARATION AND EDA PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 80)

print(f"\n📁 Output Locations:")
print(f"  Processed data: {processed_dir}")
print(f"  Plots: {plots_dir}")
print(f"\n📊 Prepared Datasets Ready for Transformation and Modeling!")